# 🗺️ 01 · MapReduce：从原理到手写词频统计

> MapReduce 是"分布式计算的启蒙思想"：**Map（分）→ Shuffle（排序分组）→ Reduce（合）**。
> 本节手写**词频统计**与**平均分计算**，并演示 **Shuffle 排序**如何产生。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | MapReduce 三个阶段？ | Map / Shuffle / Reduce |
| 2 | 怎么处理大数据？ | 分而治之：数据分片并行 |
| 3 | Shuffle 到底是什么？ | 排序 + 按键分组 |
| 4 | 典型应用？ | 词频、TopN、分组聚合 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
import sqlite3
import os
rng = np.random.default_rng(42)


## 1️⃣ 思想三句话（必背）

In [2]:
print('''1) Map：把每个输入记录变成 (key, value) 中间结果，可并行；
2) Shuffle：系统按 key 排序并分组（相同 key 发给同一 Reduce）；
3) Reduce：对一个 key 的所有 value 做聚合，输出最终结果。

核心价值：把"不可并行"的聚合问题拆成"可并行"的 map + 局部聚合。
''')

1) Map：把每个输入记录变成 (key, value) 中间结果，可并行；
2) Shuffle：系统按 key 排序并分组（相同 key 发给同一 Reduce）；
3) Reduce：对一个 key 的所有 value 做聚合，输出最终结果。

核心价值：把"不可并行"的聚合问题拆成"可并行"的 map + 局部聚合。



## 2️⃣ 构造模拟"日志数据"（分片）

In [3]:
# 模拟 3 个数据分片，每片若干行单词
splits = [
    ['apple banana apple', 'cherry apple'],
    ['banana banana cherry', 'apple'],
    ['cherry cherry banana apple', 'apple cherry'],
]
for i, s in enumerate(splits):
    print(f'分片{i}: {s}')
print('\n➡️ 真实场景：每个分片在（虚拟）不同机器上并行 Map')

分片0: ['apple banana apple', 'cherry apple']
分片1: ['banana banana cherry', 'apple']
分片2: ['cherry cherry banana apple', 'apple cherry']

➡️ 真实场景：每个分片在（虚拟）不同机器上并行 Map


## 3️⃣ Map 阶段：每个分片产出 (词, 1)

In [4]:
def mapper(line):
    out = []
    for w in line.lower().split():
        out.append((w, 1))
    return out

intermediate = []
for i, split in enumerate(splits):
    partition_out = []
    for line in split:
        partition_out += mapper(line)
    intermediate += partition_out
    print(f'分片{i} Map 输出: {partition_out}')
print(f'\n全部中间结果 {len(intermediate)} 条')
print('➡️ Map 输出也称 intermediate records，key=词，value=1')

分片0 Map 输出: [('apple', 1), ('banana', 1), ('apple', 1), ('cherry', 1), ('apple', 1)]
分片1 Map 输出: [('banana', 1), ('banana', 1), ('cherry', 1), ('apple', 1)]
分片2 Map 输出: [('cherry', 1), ('cherry', 1), ('banana', 1), ('apple', 1), ('apple', 1), ('cherry', 1)]

全部中间结果 15 条
➡️ Map 输出也称 intermediate records，key=词，value=1


## 4️⃣ Shuffle：按 key 排序与分组

In [5]:
# Shuffle = 排序（按 key） + 分组（相同 key 的 value 集合到一起）
intermediate_sorted = sorted(intermediate, key=lambda kv: kv[0])
grouped = {}
for k, v in intermediate_sorted:
    grouped.setdefault(k, []).append(v)
print('排序后（前 8 条）:', intermediate_sorted[:8])
print('\n分组结果:', {k: len(v) for k, v in grouped.items()})
print('➡️ 面试点：Shuffle 是 MR 最耗时的环节（排序+网络传输），'
      '优化思路：map 端预聚合（combiner）、压缩中间数据')

排序后（前 8 条）: [('apple', 1), ('apple', 1), ('apple', 1), ('apple', 1), ('apple', 1), ('apple', 1), ('banana', 1), ('banana', 1)]

分组结果: {'apple': 6, 'banana': 4, 'cherry': 5}
➡️ 面试点：Shuffle 是 MR 最耗时的环节（排序+网络传输），优化思路：map 端预聚合（combiner）、压缩中间数据


## 5️⃣ Reduce 阶段：聚合计数

In [6]:
result = {k: sum(v) for k, v in grouped.items()}
total = sum(result.values())
print('词频统计结果:', dict(sorted(result.items())))
print('总单词数:', total)

# 对照 pandas 验证
df = pd.DataFrame([w for s in splits for line in s for w in line.lower().split()], columns=['word'])
expected = df['word'].value_counts().sort_index().to_dict()
assert result == expected, 'MR 结果与 pandas 对照不一致'
print('✅ 与 pandas value_counts 对照完全一致')
total_words = sum(len(line.split()) for s in splits for line in s)
print('数据总单词数:', total_words)
assert total_words == 15
print('✅ MapReduce 逻辑正确')

词频统计结果: {'apple': 6, 'banana': 4, 'cherry': 5}
总单词数: 15
✅ 与 pandas value_counts 对照完全一致
数据总单词数: 15
✅ MapReduce 逻辑正确


## 6️⃣ 进阶：TopN 与均值（Reduce 变体）

In [7]:
# 平均分：Map 出 (班级, 分数)，Reduce 求和并计数
records = [('一班', 85), ('二班', 92), ('一班', 78), ('三班', 88), ('二班', 95), ('一班', 90), ('三班', 70)]
grouped_scores = {}
for cls, sc in records:
    grouped_scores.setdefault(cls, []).append(sc)
avg = {cls: round(np.mean(vs), 2) for cls, vs in grouped_scores.items()}
print('各班平均分:', avg)

# TopN：全局 top2（跨分片，可用分区+归并）
top2 = sorted((v, k) for k, v in result.items())[-2:]
print('词频 Top2:', [(k, v) for v, k in top2[::-1]])
assert len(avg) == 3
print('✅ 变体任务完成')

各班平均分: {'一班': np.float64(84.33), '二班': np.float64(93.5), '三班': np.float64(79.0)}
词频 Top2: [('apple', 6), ('cherry', 5)]
✅ 变体任务完成


## 7️⃣ MapReduce 局限（面试必问）

In [8]:
print('''1) 磁盘 IO 重：中间结果落盘，迭代计算慢；
   -> 被 Spark（内存计算）替代，仅少数批任务保留；
2) 编程模型受限：复杂 DAG/迭代算法难表达；
3) 实时性差：分钟级延迟起步；
4) 大量 shuffle 排序开销；
5) 适用场景：超大单遍聚合（离线日志统计），如今多由 Hive/Spark SQL 语法糖承接。
''')

1) 磁盘 IO 重：中间结果落盘，迭代计算慢；
   -> 被 Spark（内存计算）替代，仅少数批任务保留；
2) 编程模型受限：复杂 DAG/迭代算法难表达；
3) 实时性差：分钟级延迟起步；
4) 大量 shuffle 排序开销；
5) 适用场景：超大单遍聚合（离线日志统计），如今多由 Hive/Spark SQL 语法糖承接。



## 📝 自测清单

- [ ] 能讲清 Map/Shuffle/Reduce 三阶段
- [ ] 能手写词频统计（Map 产出 (w,1) → 分组 → 求和）
- [ ] 能解释 Shuffle 为什么慢
- [ ] 能说出 MR 五点局限与替代方案